(base-target-comparison)=
# Base vs Target Model Comparison

### model_utils.<br>**model_comparison**

```python
model_comparison(create_model_func, y_val_columns, df_train, additional_features, 
                 base_formula, target_formula, x_vals=['age', 'sex', 'site'], 
                 main_folder=None, save_log=True, num_cores=None, force_overwrite=False, 
                 save_models=False, use_multiprocessing=True, eval_data=None, 
                 affected=None, func_name=None)
```

&nbsp;&nbsp;&nbsp;&nbsp;Compares base and target GAMLSS models to determine whether additional features significantly improve model fit. Positive AIC/BIC differences indicate the target model is better.

&nbsp;&nbsp;&nbsp;&nbsp;<span style="background-color: #f5f5f5; padding: 4px 8px; display: inline-block; width: 90%;">**Parameters:**</span>

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**create_model_func** : ***callable*** <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Function to create a GAMLSS model

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**y_val_columns** : ***list of str*** <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;List of y-value column names to model (one outcome may also be given as a string, e.g. `'thickness'`). All outcomes and formulas are checked before anything is fitted, and one error lists every problem found

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**df_train** : ***pandas.DataFrame*** <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Training dataframe

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**additional_features** : ***list of str*** <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Additional features to include in the target model

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**base_formula** : ***dict*** <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Dictionary specifying the base model formula

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**target_formula** : ***dict*** <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Dictionary specifying the target model formula

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**x_vals** : ***list of str***, *default=['age', 'sex', 'site']* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;List of predictor variables

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**main_folder** : ***str***, *default='results'* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Folder for saving results and logs

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**save_log** : ***bool***, *default=True* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Whether to save logs

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**num_cores** : ***int***, *optional* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Number of CPU cores for multiprocessing (uses 4 if None)

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**force_overwrite** : ***bool***, *default=False* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Whether to overwrite existing cached results. If False, a result saved earlier in main_folder is reused only when it was made with the same settings (formulas, model code, data, number of permutations, evaluation data); results made with other settings are computed again, with a warning. The settings are saved next to each result in a small `_settings.json` file

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**save_models** : ***bool***, *default=False* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Whether to save fitted models

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**use_multiprocessing** : ***bool***, *default=True* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Whether to use multiprocessing

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**eval_data** : ***pandas.DataFrame***, *optional* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;DataFrame for evaluation

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**affected** : ***array-like***, *optional* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Array indicating affected status for evaluation. If None, assumes all subjects are healthy. It must have one value per row of eval_data (checked before anything is fitted)

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**func_name** : ***str***, *optional* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Distribution family name for evaluation (auto-detected if None)

&nbsp;&nbsp;&nbsp;&nbsp;<span style="background-color: #f5f5f5; padding: 4px 8px; display: inline-block; width: 90%;">**Returns:**</span>

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**comparison_df** : ***pandas.DataFrame*** <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;AIC and BIC values for base and target models with differences

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**summary_df** : ***pandas.DataFrame*** <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Best models and their differences for each y-value

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**eval_results** : ***dict***, *optional* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Evaluation metrics (returned if eval_data provided)

<br>

**Examples:**

In [52]:
import multiprocessing as mp

from normgamlss.gamlss_main import Gamlss
from normgamlss.model_utils import model_comparison

# This function gives us the flexibility on what we want to change and what we want to keep constant
# For example, we can keep the nu and tau formula fixed by removing it from this creat_model_func and in the next cell we can
# just change the other parameters or even change the way we want to input

def create_model_func(model_name, x_vals, y_val, mu_formula, sigma_formula, nu_formula, tau_formula):
    def model_func(data):
        gamlss_model = Gamlss(model_name, x_vals, y_val)
        gamlss_model.fit(
            r_code=f"""
            model <- gamlss(
                y ~ {mu_formula},
                sigma.formula = ~ {sigma_formula},
                nu.formula = ~ {nu_formula},
                tau.formula = ~ {tau_formula},
                family = SHASH(),
                method = mixed(10, 100)
            )
            """,
            data=data
        )
        return gamlss_model
    return model_func


In [53]:
#### Preparing the input
main_folder = r"../../results/model_comparison"
num_cores = mp.cpu_count()  # Get the number of CPU cores
y_val_columns = ['y_thickness1','y_thickness2', 'y_thickness3', 'y_thickness4']
additional_features =  ['feature_volume1', 'feature_volume2']

# Formula's for baseline and target
base_formula = {'mu_formula': 'pvc(age, by = as.factor(sex)) +  re(random=~1|site)',
                'sigma_formula': 'pvc(age) +  re(random=~1|site)',
                'nu_formula': '1',
                'tau_formula': '1'}

target_formula = {'mu_formula': 'pvc(age, by = as.factor(sex)) +  re(random=~1|site)',
                'sigma_formula': 'pvc(age) +  re(random=~1|site) + feature_volume1 + feature_volume2',
                'nu_formula': '1',
                'tau_formula': '1'}


print(f"Number of CPU cores: {num_cores}")


Number of CPU cores: 28


In [54]:
comparison_df, summary_df, eval_results = model_comparison(
    create_model_func = create_model_func,
    y_val_columns=y_val_columns,
    df_train=train_data1,
    eval_data=test_data1,
    affected = test_data1['affected'],
    additional_features=additional_features,
    base_formula=base_formula,
    target_formula=target_formula,
    x_vals=['age', 'sex', 'site'],
    main_folder= main_folder,
    save_log=True,
    num_cores=4,  # number of CPU cores to use
    force_overwrite=True,  # recompute results even if cached versions exist
    save_models=True,
    use_multiprocessing=True
)

2026-10-29 18:26:13,385 - INFO - Using multiprocessing for processing
2026-10-29 18:26:13,386 - INFO - Using 4 cores for multiprocessing
2026-10-29 18:26:13,414 - INFO - Skipping 0 already processed y_vals
2026-10-29 18:26:13,415 - INFO - Processing 4 y_vals
2026-10-29 18:26:13,595 - INFO - Running models for y_val: y_thickness1
2026-10-29 18:26:13,598 - INFO - Running models for y_val: y_thickness3
2026-10-29 18:26:13,598 - INFO - Running models for y_val: y_thickness2
2026-10-29 18:26:13,599 - INFO - Running models for y_val: y_thickness4
GAMLSS-RS iteration 1: Global Deviance = 4164.66 
GAMLSS-RS iteration 1: Global Deviance = 3594.915 
GAMLSS-RS iteration 1: Global Deviance = 3517.461 
GAMLSS-RS iteration 2: Global Deviance = 4119.447 
GAMLSS-RS iteration 1: Global Deviance = 3517.72 
GAMLSS-RS iteration 2: Global Deviance = 3509.95 
GAMLSS-RS iteration 3: Global Deviance = 4085.34 
GAMLSS-RS iteration 4: Global Deviance = 4057.893 
GAMLSS-RS iteration 3: Global Deviance = 3446.16 

In [55]:
comparison_df

,y_val,base_AIC,target_AIC,base_BIC,target_BIC,base_Converged,target_Converged,AIC_diff,BIC_diff
0,y_thickness1,3065.477334,92978.312900,3119.575818,93402.233959,True,False,-89912.835566,-90282.658141
1,y_thickness2,3087.693830,8883.398457,3152.212469,9257.898361,True,False,-5795.704627,-6105.685892
2,y_thickness3,3942.367646,3946.114319,3989.406086,4003.512769,True,True,-3.746673,-14.106683
3,y_thickness4,3094.552055,3466.409953,3144.257964,3598.277069,False,False,-371.857898,-454.019105


In [56]:
summary_df

,y_val,Best_Model_AIC,AIC_Diff,Best_Model_BIC,BIC_Diff
0,y_thickness1,model_y_thickness1_base,89912.835566,model_y_thickness1_base,90282.658141
1,y_thickness2,model_y_thickness2_base,5795.704627,model_y_thickness2_base,6105.685892
2,y_thickness3,model_y_thickness3_base,3.746673,model_y_thickness3_base,14.106683
3,y_thickness4,model_y_thickness4_base,371.857898,model_y_thickness4_base,454.019105


In [57]:
eval_results.keys()

dict_keys(['y_thickness1', 'y_thickness2', 'y_thickness3', 'y_thickness4'])

In [58]:
eval_results['y_thickness1']

,Category,base,target
Metric,,,
Z_Scores_Mean,GAMLSS,0.038786,-1.559026
Z_Scores_SD,GAMLSS,0.995145,1.999308
Logarithmic_Score,GAMLSS,-1.462897,-2.326660
Skewness,GAMLSS,-0.019956,-0.262766
Kurtosis,GAMLSS,-0.520425,-0.612559
Normality_Test_Statistic,GAMLSS,0.992989,0.920173
MAE,ML,0.943362,1.237306
MSE,ML,1.225695,2.341959
RMSE,ML,1.107111,1.530346
